# Bosphorus Ship Detection: Live Demonstration

Fine-tuned YOLO11 for Ship Detection in the Istanbul Bosphorus Strait

Recep Ertuğrul Ekşi, Üsküdar University, 2026

This notebook demonstrates the detection system developed in the thesis. It compares the fine-tuned model against the vanilla COCO-pretrained YOLO11s on the same images, and shows detection behaviour under degraded imaging conditions (blur, low light, noise, fog).

Repository: https://github.com/Mizar-107/Uskudar-University-Thesis-Repository

In [ ]:
!pip -q install ultralytics
import ultralytics, torch
print('Ultralytics', ultralytics.__version__)
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only')

In [ ]:
# Load both models: the fine-tuned weights from the thesis repository, and the vanilla COCO-pretrained YOLO11s
import urllib.request, os
from ultralytics import YOLO

WEIGHTS_URL = 'https://github.com/Mizar-107/Uskudar-University-Thesis-Repository/raw/main/bosphorus_yolo11s_best.pt'
WEIGHTS = 'bosphorus_yolo11s_best.pt'
if not os.path.exists(WEIGHTS):
    urllib.request.urlretrieve(WEIGHTS_URL, WEIGHTS)
finetuned = YOLO(WEIGHTS)
vanilla = YOLO('yolo11s.pt')
print(f'fine-tuned: {WEIGHTS} ({os.path.getsize(WEIGHTS)/1e6:.1f} MB), classes: {finetuned.names}')
print('vanilla: yolo11s.pt (COCO, 80 classes)')

In [ ]:
# Sample frames from the published bogaz_v_1 dataset (Roboflow Universe, CC BY 4.0)
import urllib.request, os
SAMPLES = {
    'sample_1.jpg': 'https://source.roboflow.com/WekdB7DHWbOtqM2giPZhADczG3j1/005CskKDDzW4REEQckOn/original.jpg',
    'sample_2.jpg': 'https://source.roboflow.com/WekdB7DHWbOtqM2giPZhADczG3j1/03VqP8OeeoNBMCgDDEkr/original.jpg',
    'sample_3.jpg': 'https://source.roboflow.com/WekdB7DHWbOtqM2giPZhADczG3j1/0GWfecZQJAhVQL20OTR8/original.jpg',
    'sample_4.jpg': 'https://source.roboflow.com/WekdB7DHWbOtqM2giPZhADczG3j1/0NGO9KkSki4t7tcpwfSN/original.jpg',
}
DEMO_IMAGES = []
for name, url in SAMPLES.items():
    if not os.path.exists(name):
        try:
            urllib.request.urlretrieve(url, name)
        except Exception as e:
            print('skip', name, e); continue
    DEMO_IMAGES.append(name)
print('demo images ready:', DEMO_IMAGES)

In [ ]:
# To run the demonstration on a new image, execute this cell and select a file
from google.colab import files
up = files.upload()
if up:
    DEMO_IMAGES = list(up.keys())
print('demo images:', DEMO_IMAGES)

## 1. Vanilla versus fine-tuned, side by side

Both models run on the same image: the vanilla COCO-pretrained YOLO11s at its default 640 input, and the fine-tuned model at 1088, as in the thesis evaluation (confidence 0.25 for both).

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import os
os.makedirs('demo_outputs', exist_ok=True)

def run(model, img_bgr, imgsz):
    r = model.predict(source=img_bgr, conf=0.25, imgsz=imgsz, verbose=False)[0]
    n = len(r.boxes)
    confs = r.boxes.conf.cpu().numpy() if n else np.array([])
    return r.plot(), n, confs

for path in DEMO_IMAGES:
    img = cv2.imread(path)
    if img is None:
        print('cannot read', path); continue
    van_img, van_n, _ = run(vanilla, img, 640)
    ft_img, ft_n, ft_c = run(finetuned, img, 1088)
    fig, axes = plt.subplots(1, 2, figsize=(18, 6))
    axes[0].imshow(cv2.cvtColor(van_img, cv2.COLOR_BGR2RGB))
    axes[0].set_title(f'Vanilla YOLO11s (COCO, 640): {van_n} detection(s)', fontsize=13)
    axes[0].axis('off')
    axes[1].imshow(cv2.cvtColor(ft_img, cv2.COLOR_BGR2RGB))
    conf_txt = f', mean conf {ft_c.mean():.2f}' if ft_n else ''
    axes[1].set_title(f'Fine-tuned YOLO11s (1088): {ft_n} ship(s){conf_txt}', fontsize=13)
    axes[1].axis('off')
    fig.suptitle(os.path.basename(path), fontsize=12)
    plt.tight_layout()
    plt.savefig(f'demo_outputs/compare_{os.path.splitext(os.path.basename(path))[0]}.png', dpi=120, bbox_inches='tight')
    plt.show()

## 2. Detection under degraded conditions

The fine-tuned model runs on the original image and on artificially degraded versions of it: blur, heavy blur, low light, sensor noise, and simulated fog.

In [ ]:
import pandas as pd

def degrade(img, mode):
    if mode == 'original':
        return img
    if mode == 'blur':
        return cv2.GaussianBlur(img, (15, 15), 0)
    if mode == 'heavy blur':
        return cv2.GaussianBlur(img, (31, 31), 0)
    if mode == 'dark':
        return np.clip(img.astype(np.float32) * 0.35, 0, 255).astype(np.uint8)
    if mode == 'noise':
        return np.clip(img.astype(np.float32) + np.random.normal(0, 25, img.shape), 0, 255).astype(np.uint8)
    if mode == 'fog':
        return cv2.addWeighted(img, 0.45, np.full_like(img, 255), 0.55, 0)
    raise ValueError(mode)

CONDITIONS = ['original', 'blur', 'heavy blur', 'dark', 'noise', 'fog']
rows = []
for path in DEMO_IMAGES:
    img = cv2.imread(path)
    if img is None:
        continue
    fig, axes = plt.subplots(2, 3, figsize=(18, 8))
    fig.suptitle(os.path.basename(path), fontsize=12)
    for ax, cond in zip(axes.ravel(), CONDITIONS):
        annotated, n, confs = run(finetuned, degrade(img, cond), 1088)
        ax.imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
        extra = f', mean conf {confs.mean():.2f}' if n else ''
        ax.set_title(f'{cond}: {n} ship(s){extra}', fontsize=11)
        ax.axis('off')
        rows.append({'image': os.path.basename(path), 'condition': cond, 'ships detected': n})
    plt.tight_layout()
    plt.savefig(f'demo_outputs/degraded_{os.path.splitext(os.path.basename(path))[0]}.png', dpi=120, bbox_inches='tight')
    plt.show()

summary = pd.DataFrame(rows).pivot(index='image', columns='condition', values='ships detected')[CONDITIONS]
summary.to_csv('demo_outputs/demo_summary.csv')
summary

In [ ]:
# Export all generated figures
import shutil
shutil.make_archive('demo_outputs_pack', 'zip', 'demo_outputs')
from google.colab import files as gfiles
gfiles.download('demo_outputs_pack.zip')